In [ ]:
# Defining database credentials

import asyncio
import psycopg2
from google import genai
import os

GOOGLE_API_KEY = "#"
PG_HOST = "#"
PG_PORT = 1234
PG_DB = "db-name"
PG_USER = "username"
PG_PASSWORD = "password"

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

print("Config loaded successfully")

Config loaded successfully


In [20]:
# Connecting to db

def get_connection():
    return psycopg2.connect(
        host=PG_HOST, port=PG_PORT,
        dbname=PG_DB, user=PG_USER, password=PG_PASSWORD
    )

def execute_ddl(sql: str) -> str:

    try:
        conn = get_connection()
        conn.autocommit = True          # critical for DDL
        cur = conn.cursor()
        cur.execute(sql)
        cur.close()
        conn.close()
        return "SUCCESS"
    except Exception as e:
        return f"ERROR: {str(e)}"
    
# Quick sanity check
test = execute_ddl("SELECT 1")          # lightweight connectivity test
print("DB connection:", test)

DB connection: SUCCESS


In [21]:
from autogen_core.models import (
    ModelInfo,
    CreateResult,
    RequestUsage,
)
from autogen_core import CancellationToken

In [22]:
class GeminiChatClient:
    def __init__(self):
        self.model_info = ModelInfo(
            vision=False,
            function_calling=False,
            json_output=False,
            family="unknown",
            structured_output=False,
        )

    @property
    def capabilities(self):
        return self.model_info

    async def create(self, messages, **kwargs) -> CreateResult:
        parts = []
        for m in messages:
            if hasattr(m, "content"):
                content = m.content
                if isinstance(content, str):
                    parts.append(content)
                elif isinstance(content, list):
                    for item in content:
                        if hasattr(item, "text"):
                            parts.append(item.text)
        prompt = "\n".join(parts)

        response = gemini_client.models.generate_content(
            model="models/gemini-flash-latest",
            contents=prompt,
        )
        text = response.text.strip()

        lines = text.split("\n")
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()

        return CreateResult(
            content=text,
            usage=RequestUsage(prompt_tokens=0, completion_tokens=0),
            finish_reason="stop",
            cached=False,
        )

    def count_tokens(self, messages, **kwargs):
        return 0

    def remaining_tokens(self, messages, **kwargs):
        return 10000


class ExecutorChatClient:
    def __init__(self):
        self.model_info = ModelInfo(
            vision=False,
            function_calling=False,
            json_output=False,
            family="unknown",
            structured_output=False,
        )

    @property
    def capabilities(self):
        return self.model_info

    async def create(self, messages, **kwargs) -> CreateResult:
        ddl = ""
        for m in reversed(messages):
            if hasattr(m, "content") and isinstance(m.content, str):
                ddl = m.content
                break

        result = execute_ddl(ddl)

        return CreateResult(
            content=result,
            usage=RequestUsage(prompt_tokens=0, completion_tokens=0),
            finish_reason="stop",
            cached=False,
        )

    def count_tokens(self, messages, **kwargs):
        return 0

    def remaining_tokens(self, messages, **kwargs):
        return 10000


print("Clients are ready")

Clients are ready


In [23]:
from autogen_agentchat.agents import AssistantAgent

# Object classifier agent

classifier = AssistantAgent(
    name="Classifier",
    model_client=GeminiChatClient(),
    system_message="""
You are a SQL Server object type classifier.
Given any SQL Server DDL snippet, output ONLY a JSON object on a single line.

Rules:
- Detect the primary object type from: TABLE, STORED_PROCEDURE, FUNCTION_SCALAR,
  FUNCTION_TABLE_VALUED, TRIGGER, VIEW
- Extract the object name and (for triggers) the target table name.

Output format (exactly, no extra text):
{"type": "TABLE", "name": "StaffMembers", "target_table": null}
{"type": "TRIGGER", "name": "trg_salary_change", "target_table": "StaffMembers"}
{"type": "STORED_PROCEDURE", "name": "GetActiveStaff", "target_table": null}

Output ONLY the JSON line. Nothing else.
""",
)

print("Classifier agent initiated")

# Translator agent

translator = AssistantAgent(
    name="Translator",
    model_client=GeminiChatClient(),
    system_message="""
You are a SQL Server to PostgreSQL migration expert.
You will receive the SQL Server code prefixed with a classification tag like:
  [OBJECT TYPE: TRIGGER | name: trg_salary_change | target_table: StaffMembers]

Apply ONLY the rules for that object type. Ignore all other sections.

== TABLE ==
- IDENTITY(1,1)       → GENERATED ALWAYS AS IDENTITY
- NVARCHAR(n)         → VARCHAR(n),  NVARCHAR(MAX) → TEXT
- DATETIME            → TIMESTAMP,   GETDATE()     → NOW()
- MONEY               → NUMERIC(19,4)
- BIT                 → BOOLEAN,  default 1 → TRUE,  default 0 → FALSE
- Remove [ ] brackets

== STORED_PROCEDURE ==
- CREATE PROCEDURE          → CREATE OR REPLACE PROCEDURE
- @param_name TYPE          → IN param_name TYPE  (in signature)
- AS BEGIN…END              → LANGUAGE plpgsql AS $$ BEGIN…END $$
- Remove SET NOCOUNT ON
- PRINT 'x'                 → RAISE NOTICE 'x'
- ISNULL(x,y)               → COALESCE(x,y)
- @@ROWCOUNT                → GET DIAGNOSTICS n = ROW_COUNT
- Prefix all parameter names with p_ to avoid column name collisions
  e.g. IN p_DeptID INT, then use p_DeptID in the body
- Apply TABLE data type rules


== FUNCTION_SCALAR ==
- CREATE FUNCTION           → CREATE OR REPLACE FUNCTION
- Remove @ prefix from params
- Keep RETURNS <type>; wrap body in $$ … $$ LANGUAGE plpgsql
- str1 + str2               → str1 || str2
- Prefix all parameter names with p_ to avoid column name collisions
  e.g. IN p_DeptID INT, then use p_DeptID in the body
- Apply TABLE data type rules

== FUNCTION_TABLE_VALUED ==
- Same as FUNCTION_SCALAR header rules
- RETURNS TABLE AS RETURN(…) → RETURNS TABLE(col type, …) with RETURN QUERY
- Prefix all parameter names with p_ to avoid column name collisions
  e.g. IN p_DeptID INT, then use p_DeptID in the body
- Apply TABLE data type rules

== TRIGGER ==
- Split into TWO objects:
  1. CREATE OR REPLACE FUNCTION <name>_fn() RETURNS TRIGGER LANGUAGE plpgsql AS $$
       BEGIN
         … body (INSERTED→NEW, DELETED→OLD, IF UPDATE(col)→IF NEW.col<>OLD.col) …
         RETURN NEW;
       END; $$;
  2. CREATE TRIGGER <name>
       AFTER INSERT OR UPDATE OR DELETE ON <target_table>
       FOR EACH ROW EXECUTE FUNCTION <name>_fn();
- Apply TABLE data type rules

== VIEW ==
- CREATE VIEW               → CREATE OR REPLACE VIEW
- Remove WITH SCHEMABINDING
- Remove WITH (NOLOCK)
- Remove dbo. prefixes
- TOP n                     → LIMIT n
- ISNULL(x,y)               → COALESCE(x,y)
- Apply TABLE data type rules

Return ONLY the converted SQL. No explanations. No markdown fences.
""",
)

print("Translator agent initiated")

# Pre executor for dropping objects before executing

pre_executor = AssistantAgent(
    name="PreExecutor",
    model_client=GeminiChatClient(),
    system_message="""
You are a PostgreSQL cleanup helper.
You will receive a classification tag followed by PostgreSQL DDL, like:
  [OBJECT TYPE: STORED_PROCEDURE | name: GetActiveStaff]

  CREATE OR REPLACE PROCEDURE GetActiveStaff ...

=== CRITICAL RULES ===
1. ONLY generate DROP statements for the EXACT object type in the [OBJECT TYPE] tag.
2. NEVER drop or recreate any other object types not mentioned in the tag.
3. NEVER add any CREATE statements that are not already in the input SQL.
4. ONLY prepend the DROP statement(s). Leave the rest of the SQL completely untouched.
======================

DROP statement to prepend by object type:

- TABLE:
    DROP TABLE IF EXISTS <name> CASCADE;
    (if multiple tables, drop child/dependent tables first, then parent)

- STORED_PROCEDURE:
    DROP PROCEDURE IF EXISTS <name>(<arg types>) CASCADE;

- FUNCTION_SCALAR or FUNCTION_TABLE_VALUED:
    DROP FUNCTION IF EXISTS <name>(<arg types>) CASCADE;

- TRIGGER:
    DROP TRIGGER IF EXISTS <name> ON <target_table>;
    DROP FUNCTION IF EXISTS <name>_fn() CASCADE;
    (always drop trigger before its function)

- VIEW:
    DROP VIEW IF EXISTS <name>;

Output format (strictly follow this):
  <DROP statement(s) for the tagged type only>
  <blank line>
  <original input SQL, completely unchanged>

Return ONLY this. No explanations. No markdown fences. No extra CREATE statements.
""",
)

print("Pre-executor agent initiated")

# Executor agent

executor = AssistantAgent(
    name="Executor",
    model_client=ExecutorChatClient(),
    system_message="Execute the given PostgreSQL DDL and return SUCCESS or the full error message.",
)

print("Executor agent initiated")

# Self corrector agent

self_corrector = AssistantAgent(
    name="SelfCorrector",
    model_client=GeminiChatClient(),
    system_message="""
You are a PostgreSQL expert who fixes broken SQL.
You will receive a classification tag, the broken code, and the error message.

Apply ONLY the fix strategies for the classified type:

== TABLE ==
- Add IF NOT EXISTS if duplicate table error
- Remove FK constraint if referenced table doesn't exist yet
- Fix CHECK constraint column names

== STORED_PROCEDURE ==
- Fix $$ … $$ block syntax
- Fix IN parameter declarations
- Add missing LANGUAGE plpgsql

== FUNCTION_SCALAR / FUNCTION_TABLE_VALUED ==
- Fix RETURNS type mismatch
- Add missing RETURN / RETURN QUERY
- Fix $$ delimiter issues

== TRIGGER ==
- Ensure BOTH the trigger function AND CREATE TRIGGER are present
- Add missing RETURN NEW at end of trigger function
- Use EXECUTE FUNCTION (not EXECUTE PROCEDURE) for Postgres 14+
- Fix function name mismatch between CREATE TRIGGER and the function definition

== VIEW ==
- Fix column ambiguity (add table aliases)
- Remove any remaining dbo. or [bracket] syntax

Return ONLY the corrected SQL. No explanations. No markdown fences.
""",
)

print("Self-corrector agent initiated")

# Validator agent

validator = AssistantAgent(
    name="Validator",
    model_client=GeminiChatClient(),
    system_message="""
You are a PostgreSQL code reviewer.
You will receive a classification tag and the SQL to validate.

Check ONLY the rules for the classified type:

== ALL TYPES (always check) ==
- No SQL Server remnants: IDENTITY, NVARCHAR, DATETIME, GETDATE, ISNULL, NOLOCK, SCHEMABINDING, dbo.
- No square brackets [ ] around identifiers

== TABLE ==
- Correct PG data types, GENERATED ALWAYS AS IDENTITY if needed

== STORED_PROCEDURE ==
- $$ … $$ LANGUAGE plpgsql block present
- Parameters use IN keyword, no @ prefixes
- No SET NOCOUNT ON

== FUNCTION_SCALAR / FUNCTION_TABLE_VALUED ==
- Explicit RETURNS type declared
- LANGUAGE plpgsql present
- TABLE-valued: uses RETURNS TABLE(…) and RETURN QUERY

== TRIGGER ==
- Trigger function present with RETURNS TRIGGER
- Ends with RETURN NEW or RETURN OLD
- CREATE TRIGGER uses EXECUTE FUNCTION (not EXECUTE PROCEDURE)
- Both objects (function + trigger) are present

== VIEW ==
- No SCHEMABINDING, NOLOCK, or dbo. remaining

IMPORTANT:
- If all checks pass → reply with exactly: VALID
- If any check fails → reply with exactly: INVALID: <specific reason>
- Do NOT say VALID or INVALID anywhere in your reasoning — only in your final answer
""",
)

print("Validator agent initiated")

Classifier agent initiated
Translator agent initiated
Pre-executor agent initiated
Executor agent initiated
Self-corrector agent initiated
Validator agent initiated


In [24]:
import json

async def run_migration(sql_server_code: str, max_retries: int = 3) -> str | None:
    print("=" * 55)
    print("  SQL to Postgres Pipeline  (v2)")
    print("=" * 55)

    # STEP 1: Classify
    print("\n[1] Classifying object type...")
    result = await classifier.run(task=sql_server_code)
    raw_tag = result.messages[-1].content.strip()
    print(f"    Classification: {raw_tag}")

    try:
        meta = json.loads(raw_tag)
        obj_type   = meta.get("type", "UNKNOWN")
        obj_name   = meta.get("name", "unknown")
        target_tbl = meta.get("target_table") or ""
        type_tag   = (
            f"[OBJECT TYPE: {obj_type} | name: {obj_name}"
            + (f" | target_table: {target_tbl}" if target_tbl else "")
            + "]"
        )
    except json.JSONDecodeError:
        # Graceful fallback if model adds extra text
        type_tag = f"[OBJECT TYPE: UNKNOWN]"
        obj_type = "UNKNOWN"
        print("    Warning: could not parse classification JSON, proceeding without tag.")

    # STEP 2: Translate (with type tag injected)
    print(f"\n[2] Translating ({obj_type})...")
    tagged_input = f"{type_tag}\n\n{sql_server_code}"
    result = await translator.run(task=tagged_input)
    pg_code = result.messages[-1].content.strip()
    print(f"\n    Translated:\n{pg_code}\n")

    # STEP 3: Pre-executor cleanup (auto DROP)
    print("[3] Generating idempotent cleanup prefix...")
    result = await pre_executor.run(task=f"{type_tag}\n\n{pg_code}")
    pg_code_with_drops = result.messages[-1].content.strip()
    print(f"\n    With DROPs:\n{pg_code_with_drops}\n")

    current_code = pg_code_with_drops

    # STEP 4+5: Execute with type-aware self-correction loop
    for attempt in range(1, max_retries + 1):
        print(f"[4] Executing (attempt {attempt}/{max_retries})...")
        result = await executor.run(task=current_code)
        exec_result = result.messages[-1].content
        print(f"    Result: {exec_result}\n")

        if exec_result.startswith("SUCCESS"):
            break

        if attempt == max_retries:
            print("Migration FAILED — max retries reached.")
            return None

        print("    Execution failed → routing to SelfCorrector...")
        fix_prompt = (
            f"{type_tag}\n\n"
            f"The following PostgreSQL code failed:\n\n{current_code}\n\n"
            f"Error:\n{exec_result}\n\n"
            f"Return only the corrected SQL."
        )
        result = await self_corrector.run(task=fix_prompt)
        current_code = result.messages[-1].content.strip()
        print(f"    Corrected:\n{current_code}\n")

    # STEP 6: Validate (type-scoped)
    print("[5] Validating...")
    result = await validator.run(task=f"{type_tag}\n\n{current_code}")
    validation = result.messages[-1].content.strip()
    print(f"    Validator: {validation}\n")

    if validation.startswith("VALID"):
        print("Migration COMPLETE!")
        return current_code
    else:
        # One more correction pass on validation failure before giving up
        print(f"    Validation issue: {validation} — attempting one final correction...")
        fix_prompt = (
            f"{type_tag}\n\n"
            f"This PostgreSQL code passed execution but failed validation:\n\n{current_code}\n\n"
            f"Validation error: {validation}\n\n"
            f"Return only the corrected SQL."
        )
        result = await self_corrector.run(task=fix_prompt)
        current_code = result.messages[-1].content.strip()

        result = await validator.run(task=f"{type_tag}\n\n{current_code}")
        validation = result.messages[-1].content.strip()
        if validation.startswith("VALID"):
            print("Migration COMPLETE (after post-validation fix)!")
            return current_code
        else:
            print(f"Migration FAILED — validation: {validation}")
            return None

In [28]:
# 1. TABLES (two dependent tables)
test_table = """
CREATE TABLE StaffDepartments (
    DeptID     INT           IDENTITY(1,1) PRIMARY KEY,
    DeptName   NVARCHAR(100) NOT NULL,
    Location   NVARCHAR(100)
);

CREATE TABLE StaffMembers (
    StaffID       INT           IDENTITY(1,1) PRIMARY KEY,
    FirstName     NVARCHAR(50)  NOT NULL,
    LastName      NVARCHAR(50)  NOT NULL,
    DeptID        INT,
    MonthlySalary MONEY         NOT NULL,
    IsActive      BIT           DEFAULT 1,
    JoinDate      DATETIME      DEFAULT GETDATE(),
    FOREIGN KEY (DeptID) REFERENCES StaffDepartments(DeptID)
);
"""

print("\n>>> TABLE TEST")
result_table = await run_migration(test_table)


>>> TABLE TEST
  SQL to Postgres Pipeline  (v2)

[1] Classifying object type...
    Classification: {"type": "TABLE", "name": "StaffDepartments", "target_table": null}

[2] Translating (TABLE)...

    Translated:
CREATE TABLE StaffDepartments (
    DeptID     INT           GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    DeptName   VARCHAR(100) NOT NULL,
    Location   VARCHAR(100)
);

CREATE TABLE StaffMembers (
    StaffID       INT           GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    FirstName     VARCHAR(50)  NOT NULL,
    LastName      VARCHAR(50)  NOT NULL,
    DeptID        INT,
    MonthlySalary NUMERIC(19,4) NOT NULL,
    IsActive      BOOLEAN       DEFAULT TRUE,
    JoinDate      TIMESTAMP      DEFAULT NOW(),
    FOREIGN KEY (DeptID) REFERENCES StaffDepartments(DeptID)
);

[3] Generating idempotent cleanup prefix...

    With DROPs:
DROP TABLE IF EXISTS StaffMembers CASCADE;
DROP TABLE IF EXISTS StaffDepartments CASCADE;

CREATE TABLE StaffDepartments (
    DeptID     INT   

In [29]:
# 2. STORED PROCEDURE
test_sp = """
CREATE PROCEDURE GetActiveStaff
    @DeptID INT
AS
BEGIN
    SET NOCOUNT ON;
    SELECT StaffID, FirstName, LastName
    FROM StaffMembers
    WHERE DeptID = @DeptID AND IsActive = 1;
END
"""

print("\n>>> STORED PROCEDURE TEST")
result_sp = await run_migration(test_sp)


>>> STORED PROCEDURE TEST
  SQL to Postgres Pipeline  (v2)

[1] Classifying object type...
    Classification: {"type": "STORED_PROCEDURE", "name": "GetActiveStaff", "target_table": null}

[2] Translating (STORED_PROCEDURE)...

    Translated:
CREATE OR REPLACE PROCEDURE GetActiveStaff(IN p_DeptID INT)
LANGUAGE plpgsql AS $$
BEGIN
    SELECT StaffID, FirstName, LastName
    FROM StaffMembers
    WHERE DeptID = p_DeptID AND IsActive = TRUE;
END;
$$;

[3] Generating idempotent cleanup prefix...

    With DROPs:
DROP PROCEDURE IF EXISTS GetActiveStaff(INT) CASCADE;

CREATE OR REPLACE PROCEDURE GetActiveStaff(IN p_DeptID INT)
LANGUAGE plpgsql AS $$
BEGIN
    SELECT StaffID, FirstName, LastName
    FROM StaffMembers
    WHERE DeptID = p_DeptID AND IsActive = TRUE;
END;
$$;

[4] Executing (attempt 1/3)...
    Result: SUCCESS

[5] Validating...
    Validator: VALID

Migration COMPLETE!


In [30]:
# 3. SCALAR-VALUED FUNCTION
test_function_scalar = """
CREATE FUNCTION GetFullName(
    @FirstName NVARCHAR(50),
    @LastName  NVARCHAR(50)
)
RETURNS NVARCHAR(100)
AS
BEGIN
    RETURN @FirstName + ' ' + @LastName
END
"""

print("\n>>> SCALAR FUNCTION TEST")
result_function_scalar = await run_migration(test_function_scalar)


>>> SCALAR FUNCTION TEST
  SQL to Postgres Pipeline  (v2)

[1] Classifying object type...
    Classification: {"type": "FUNCTION_SCALAR", "name": "GetFullName", "target_table": null}

[2] Translating (FUNCTION_SCALAR)...

    Translated:
CREATE OR REPLACE FUNCTION GetFullName(
    p_FirstName VARCHAR(50),
    p_LastName  VARCHAR(50)
)
RETURNS VARCHAR(100)
LANGUAGE plpgsql AS $$
BEGIN
    RETURN p_FirstName || ' ' || p_LastName;
END; $$;

[3] Generating idempotent cleanup prefix...

    With DROPs:
DROP FUNCTION IF EXISTS GetFullName(VARCHAR, VARCHAR) CASCADE;

CREATE OR REPLACE FUNCTION GetFullName(
    p_FirstName VARCHAR(50),
    p_LastName  VARCHAR(50)
)
RETURNS VARCHAR(100)
LANGUAGE plpgsql AS $$
BEGIN
    RETURN p_FirstName || ' ' || p_LastName;
END; $$;

[4] Executing (attempt 1/3)...
    Result: SUCCESS

[5] Validating...
    Validator: VALID

Migration COMPLETE!


In [31]:
# 4. TABLE-VALUED FUNCTION
test_function_tvf = """
CREATE FUNCTION GetStaffByDept(
    @DeptID INT
)
RETURNS TABLE
AS
RETURN (
    SELECT StaffID, FirstName, LastName, MonthlySalary
    FROM StaffMembers
    WHERE DeptID = @DeptID
      AND IsActive = 1
)
"""

print("\n>>> TABLE-VALUED FUNCTION TEST")
result_function_tvf = await run_migration(test_function_tvf)


>>> TABLE-VALUED FUNCTION TEST
  SQL to Postgres Pipeline  (v2)

[1] Classifying object type...
    Classification: {"type": "FUNCTION_TABLE_VALUED", "name": "GetStaffByDept", "target_table": null}

[2] Translating (FUNCTION_TABLE_VALUED)...

    Translated:
CREATE OR REPLACE FUNCTION GetStaffByDept(
    p_DeptID INT
)
RETURNS TABLE(
    StaffID INT,
    FirstName VARCHAR(50),
    LastName VARCHAR(50),
    MonthlySalary NUMERIC(19,4)
)
LANGUAGE plpgsql AS $$
BEGIN
    RETURN QUERY
    SELECT StaffID, FirstName, LastName, MonthlySalary
    FROM StaffMembers
    WHERE DeptID = p_DeptID
      AND IsActive = TRUE;
END; $$;

[3] Generating idempotent cleanup prefix...

    With DROPs:
DROP FUNCTION IF EXISTS GetStaffByDept(INT) CASCADE;

CREATE OR REPLACE FUNCTION GetStaffByDept(
    p_DeptID INT
)
RETURNS TABLE(
    StaffID INT,
    FirstName VARCHAR(50),
    LastName VARCHAR(50),
    MonthlySalary NUMERIC(19,4)
)
LANGUAGE plpgsql AS $$
BEGIN
    RETURN QUERY
    SELECT StaffID, FirstName

In [36]:
# 5. TRIGGER
test_trigger = """
CREATE TRIGGER trg_SalaryChange
ON StaffMembers
AFTER UPDATE
AS
BEGIN
    IF UPDATE(MonthlySalary)
    BEGIN
        INSERT INTO AuditLog(StaffID, Note)
        SELECT StaffID, 'Salary changed'
        FROM INSERTED;
    END
END
"""

print("\n>>> TRIGGER TEST")
result_trigger = await run_migration(test_trigger)


>>> TRIGGER TEST
  SQL to Postgres Pipeline  (v2)

[1] Classifying object type...
    Classification: {"type": "TRIGGER", "name": "trg_SalaryChange", "target_table": "StaffMembers"}

[2] Translating (TRIGGER)...

    Translated:
CREATE OR REPLACE FUNCTION trg_SalaryChange_fn() RETURNS TRIGGER LANGUAGE plpgsql AS $$
BEGIN
    IF NEW.MonthlySalary <> OLD.MonthlySalary THEN
        INSERT INTO AuditLog(StaffID, Note)
        VALUES (NEW.StaffID, 'Salary changed');
    END IF;
    RETURN NEW;
END; $$;

CREATE TRIGGER trg_SalaryChange
AFTER UPDATE ON StaffMembers
FOR EACH ROW EXECUTE FUNCTION trg_SalaryChange_fn();

[3] Generating idempotent cleanup prefix...

    With DROPs:
DROP TRIGGER IF EXISTS trg_SalaryChange ON StaffMembers;
DROP FUNCTION IF EXISTS trg_SalaryChange_fn() CASCADE;

CREATE OR REPLACE FUNCTION trg_SalaryChange_fn() RETURNS TRIGGER LANGUAGE plpgsql AS $$
BEGIN
    IF NEW.MonthlySalary <> OLD.MonthlySalary THEN
        INSERT INTO AuditLog(StaffID, Note)
        VALUES (N

In [37]:
# 6. VIEW
test_view = """
CREATE VIEW ActiveStaffView
WITH SCHEMABINDING
AS
SELECT
    s.[StaffID],
    s.[FirstName],
    s.[LastName],
    d.[DeptName],
    s.[MonthlySalary]
FROM dbo.[StaffMembers] s WITH (NOLOCK)
INNER JOIN dbo.[StaffDepartments] d
    ON s.[DeptID] = d.[DeptID]
WHERE s.[IsActive] = 1
"""

print("\n>>> VIEW TEST")
result_view = await run_migration(test_view)


>>> VIEW TEST
  SQL to Postgres Pipeline  (v2)

[1] Classifying object type...
    Classification: {"type": "VIEW", "name": "ActiveStaffView", "target_table": null}

[2] Translating (VIEW)...

    Translated:
CREATE OR REPLACE VIEW ActiveStaffView
AS
SELECT
    s.StaffID,
    s.FirstName,
    s.LastName,
    d.DeptName,
    s.MonthlySalary
FROM StaffMembers s
INNER JOIN StaffDepartments d
    ON s.DeptID = d.DeptID
WHERE s.IsActive = TRUE;

[3] Generating idempotent cleanup prefix...

    With DROPs:
DROP VIEW IF EXISTS ActiveStaffView;

CREATE OR REPLACE VIEW ActiveStaffView
AS
SELECT
    s.StaffID,
    s.FirstName,
    s.LastName,
    d.DeptName,
    s.MonthlySalary
FROM StaffMembers s
INNER JOIN StaffDepartments d
    ON s.DeptID = d.DeptID
WHERE s.IsActive = TRUE;

[4] Executing (attempt 1/3)...
    Result: SUCCESS

[5] Validating...
    Validator: VALID

Migration COMPLETE!
